# Hybrid VAE-Based Cyber Intrusion Detection in Rail Track Geometry from SCADA Systems
### A Spatial-Temporal Deep Learning Framework
**Authors:** Benjamin Boateng  
**Target:** Transportation Research Part C / IEEE TITS

---
This notebook implements two complementary detection frameworks:
- **Part 1 — Static Framework**: IF, VAE, VAE+MAD on a single-inspection snapshot (WRM_F.csv)
- **Part 2 — Temporal Framework**: LSTM-VAE on 28 monthly repeated inspections (RawD.xlsx)
- **Part 3 — Comparison**: Static vs Temporal under persistent and one-shot FDI attacks


## Section 0: Environment Setup

In [ ]:
# Install / upgrade key dependencies
!pip install -q tensorflow scikit-learn pandas numpy matplotlib seaborn scipy openpyxl statsmodels


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings, os, json
warnings.filterwarnings('ignore')

# ── Deep learning ──────────────────────────────────────────
import tensorflow as tf
import keras
import keras.ops as kops  # Keras 3 backend-agnostic ops
from tensorflow.keras.layers import (Input, Dense, Lambda, LSTM, RepeatVector,
                                      TimeDistributed, Dropout, BatchNormalization)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras import backend as K

# ── ML / stats ─────────────────────────────────────────────
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.model_selection import KFold
from sklearn.metrics import (confusion_matrix, classification_report,
                              precision_score, recall_score, f1_score,
                              roc_auc_score, roc_curve)
import statsmodels.stats.contingency_tables as smct
from scipy import stats

# ── Reproducibility ────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow {tf.__version__} | GPU: {tf.config.list_physical_devices('GPU')}")


## Section 1: Load Data from Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── Update these paths if your files are in a sub-folder ───
BASE_PATH = '/content/drive/My Drive/'   # <-- change if needed
STATIC_FILE  = BASE_PATH + 'WRM_F.csv'
TEMPORAL_FILE = BASE_PATH + 'RawD.xlsx'


---
## PART 1 — STATIC ANALYSIS (WRM_F)
### Section 2: Load & Preprocess Static Data


In [ ]:
# ── Load raw data ──────────────────────────────────────────
raw_df = pd.read_csv(STATIC_FILE)

# Rename BFI label columns only — do NOT rename BFI Value_1/2
# (renaming both creates duplicate column names, breaking pd.to_numeric)
raw_df = raw_df.rename(columns={
    'BFI L':   'BFI_L_Cat',      # category label — not used
    'BFI C ':  'BFI_C_Cat',      # category label — not used
    'BFI R ':  'BFI_R_Value',    # numeric BFI right value
    'BFI Value_1': 'BFI_L_Value', # numeric BFI left value
    'BFI Value_2': 'BFI_C_Value'  # numeric BFI centre value
})
print(f'Raw shape: {raw_df.shape}')

# ── Feature selection (domain knowledge + variance filter) ─
STATIC_FEATURES = ['LProf62','RProf62','LAlign62','RAlign62','Gage',
                   'Warp62','BFI_L_Value','BFI_C_Value','BFI_R_Value',
                   'Crosslevel','TotalLatForce','Left Cant','Right Cant','Speed']

# Safety check: no duplicate column names
dupes = [c for c in STATIC_FEATURES if raw_df.columns.tolist().count(c) > 1]
if dupes:
    raise ValueError(f'Duplicate columns detected: {dupes}. Check rename dict.')

# Coerce numeric, drop nulls
for col in STATIC_FEATURES:
    raw_df[col] = pd.to_numeric(raw_df[col], errors='coerce')
df_static = raw_df[STATIC_FEATURES].dropna().reset_index(drop=True)
print(f'After feature selection & dropna: {df_static.shape}')
print(df_static.describe().round(3))


In [ ]:
# ── Remove highly correlated features (|r| > 0.85) ─────────
corr = df_static.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop = [c for c in upper.columns if any(upper[c] > 0.85)]
df_static = df_static.drop(columns=to_drop)
print(f"Dropped (high correlation): {to_drop}")
print(f"Final features ({df_static.shape[1]}): {list(df_static.columns)}")

STATIC_COLS = list(df_static.columns)


In [ ]:
# ── Scale ──────────────────────────────────────────────────
scaler_static = StandardScaler()
X_static = scaler_static.fit_transform(df_static)
X_static_df = pd.DataFrame(X_static, columns=STATIC_COLS)
print("Scaled shape:", X_static_df.shape)

# ── Quick EDA ──────────────────────────────────────────────
fig, axes = plt.subplots(2, len(STATIC_COLS)//2 + len(STATIC_COLS)%2,
                          figsize=(18, 5))
for ax, col in zip(axes.ravel(), STATIC_COLS):
    ax.hist(df_static[col], bins=40, edgecolor='none', alpha=0.7)
    ax.set_title(col, fontsize=8); ax.set_yticks([])
for ax in axes.ravel()[len(STATIC_COLS):]:
    ax.set_visible(False)
plt.suptitle("Feature Distributions — Clean Static Data", y=1.02)
plt.tight_layout(); plt.show()


### Section 3: Static Model Definitions (IF, VAE, VAE+MAD)

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  VAE Architecture — Keras 3 / TF 2.16+
#  Fix: keras.random.normal (kops.random_normal does not exist in Keras 3)
# ─────────────────────────────────────────────────────────────────

class Sampling(keras.layers.Layer):
    """Reparametrisation trick: z = mu + exp(0.5 * log_var) * epsilon."""
    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch   = kops.shape(z_mean)[0]
        dim     = kops.shape(z_mean)[1]
        epsilon = keras.random.normal(shape=(batch, dim))   # Keras 3 correct API
        return z_mean + kops.exp(0.5 * z_log_var) * epsilon


class VAEModel(keras.Model):
    """VAE wrapper — ELBO loss via self.add_loss() so .fit() works cleanly."""
    def __init__(self, encoder, decoder, **kwargs):
        super().__init__(**kwargs)
        self._enc = encoder
        self._dec = decoder

    def call(self, x, training=False):
        z_mean, z_log_var, z = self._enc(x, training=training)
        x_hat  = self._dec(z, training=training)
        recon  = kops.mean(kops.sum(kops.square(x - x_hat), axis=1))
        kl     = -0.5 * kops.mean(
                     kops.sum(1 + z_log_var - kops.square(z_mean)
                              - kops.exp(z_log_var), axis=1))
        self.add_loss(recon + kl)
        return x_hat

    def get_config(self):
        cfg = super().get_config()
        cfg.update({
            'encoder': keras.saving.serialize_keras_object(self._enc),
            'decoder': keras.saving.serialize_keras_object(self._dec),
        })
        return cfg

    @classmethod
    def from_config(cls, config):
        enc = keras.saving.deserialize_keras_object(config.pop('encoder'))
        dec = keras.saving.deserialize_keras_object(config.pop('decoder'))
        return cls(encoder=enc, decoder=dec, **config)


def build_vae(input_dim, latent_dim=7):
    # ── Encoder ──────────────────────────────────────────────
    enc_input  = keras.Input(shape=(input_dim,), name='enc_input')
    h          = keras.layers.Dense(128, activation='relu')(enc_input)
    h          = keras.layers.BatchNormalization()(h)
    h          = keras.layers.Dropout(0.2)(h)
    h          = keras.layers.Dense(64, activation='relu')(h)
    h          = keras.layers.BatchNormalization()(h)
    z_mean     = keras.layers.Dense(latent_dim, name='z_mean')(h)
    z_log_var  = keras.layers.Dense(latent_dim, name='z_log_var')(h)
    z          = Sampling()([z_mean, z_log_var])
    encoder    = keras.Model(enc_input, [z_mean, z_log_var, z], name='encoder')

    # ── Decoder ──────────────────────────────────────────────
    dec_input  = keras.Input(shape=(latent_dim,), name='dec_input')
    d          = keras.layers.Dense(64, activation='relu')(dec_input)
    d          = keras.layers.BatchNormalization()(d)
    d          = keras.layers.Dropout(0.2)(d)
    d          = keras.layers.Dense(128, activation='relu')(d)
    dec_output = keras.layers.Dense(input_dim, activation='linear')(d)
    decoder    = keras.Model(dec_input, dec_output, name='decoder')

    vae = VAEModel(encoder, decoder, name='vae')
    vae.compile(optimizer=keras.optimizers.Adam(1e-3))
    return vae, encoder, decoder


# ─────────────────────────────────────────────────────────────────
#  Reconstruction + scoring helpers (leak-free)
#
#  - mad_score_from_train : MAD calibrated on CLEAN TRAINING errors only
#  - train_percentile_threshold : threshold from training distribution only
#  - cv_threshold REMOVED (used test labels — leakage)
# ─────────────────────────────────────────────────────────────────

def vae_reconstruction_error(encoder, decoder, X, feature_weights=None):
    _, _, z = encoder.predict(X, verbose=0)
    X_hat   = decoder.predict(z, verbose=0)
    if feature_weights is not None:
        return np.mean(feature_weights * (X - X_hat) ** 2, axis=1)
    return np.mean((X - X_hat) ** 2, axis=1)

def mad_score_from_train(train_errors, test_errors):
    """MAD calibrated on CLEAN TRAINING errors only — no leakage."""
    median = np.median(train_errors)
    mad    = np.median(np.abs(train_errors - median))
    return np.abs(test_errors - median) / (mad + 1e-8)

def train_percentile_threshold(train_scores, percentile=95):
    """Threshold derived from training distribution only — no label leakage."""
    return np.percentile(train_scores, percentile)

def evaluate(y_true, y_pred, label='Model'):
    p   = precision_score(y_true, y_pred, zero_division=0)
    r   = recall_score(y_true, y_pred, zero_division=0)
    f1  = f1_score(y_true, y_pred, zero_division=0)
    acc = np.mean(y_true == y_pred)
    print(f'{label:50s}  P={p:.3f}  R={r:.3f}  F1={f1:.3f}  Acc={acc:.3f}')
    return {'label': label, 'precision': p, 'recall': r, 'f1': f1, 'accuracy': acc}

def plot_confusion(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(4,3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Normal','Anomaly'],
                yticklabels=['Normal','Anomaly'])
    plt.title(title); plt.ylabel('True'); plt.xlabel('Predicted')
    plt.tight_layout(); plt.show()


In [ ]:
# Helper functions are defined in Cell 11:
#   vae_reconstruction_error, mad_score_from_train,
#   train_percentile_threshold, evaluate, plot_confusion
# This cell is intentionally left as a placeholder to preserve cell numbering.
pass


### Section 4: Pre-Injection Baseline (Static Clean Data)

In [ ]:
input_dim_static = X_static.shape[1]

# ── Feature weights: emphasise safety-critical sensors ──────
fw_static = np.array([1.5 if c in ('Gage','LAlign62','RAlign62','Warp62','Crosslevel')
                       else 1.0 for c in STATIC_COLS])

# ── Train VAE on clean data ──────────────────────────────────
vae_s, enc_s, dec_s = build_vae(input_dim_static, latent_dim=7)
cb = [EarlyStopping(patience=10, restore_best_weights=True),
      ReduceLROnPlateau(patience=5, factor=0.5, verbose=0)]
history = vae_s.fit(X_static, X_static,
                    epochs=100, batch_size=64,
                    validation_split=0.15,
                    callbacks=cb, verbose=0)
print("VAE training complete. Final val loss:", round(history.history['val_loss'][-1], 4))

# ── Training curve ───────────────────────────────────────────
plt.figure(figsize=(7,3))
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend()
plt.title('Static VAE — Training Curve'); plt.tight_layout(); plt.show()

In [ ]:
# ── Isolation Forest on clean data ──────────────────────────
iso_s = IsolationForest(contamination=0.05, random_state=SEED)
iso_s.fit(X_static)

# ── Pre-injection reconstruction errors ─────────────────────
err_vae_pre    = vae_reconstruction_error(enc_s, dec_s, X_static, fw_static)
iso_scores_pre = -iso_s.score_samples(X_static)

fig, axes = plt.subplots(1, 2, figsize=(11, 3))
for ax, scores, name in zip(axes,
    [iso_scores_pre, err_vae_pre],
    ['Isolation Forest', 'VAE Reconstruction Error']):
    ax.hist(scores, bins=60, alpha=0.7, edgecolor='none')
    ax.set_title(f'{name} — Pre-FDI')
    ax.set_xlabel('Score')
plt.tight_layout(); plt.show()


In [ ]:
# ── PCA & t-SNE on clean latent space ───────────────────────
z_pre = enc_s.predict(X_static, verbose=0)[0]   # z_mean

pca   = PCA(n_components=2)
z_pca = pca.fit_transform(z_pre)

tsne   = TSNE(n_components=2, random_state=SEED, perplexity=30)
z_tsne = tsne.fit_transform(z_pre[:3000])   # subsample for speed

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(z_pca[:,0], z_pca[:,1], s=2, alpha=0.3, c='steelblue')
axes[0].set_title('PCA — Latent Space (Pre-FDI)')
axes[1].scatter(z_tsne[:,0], z_tsne[:,1], s=2, alpha=0.3, c='darkorange')
axes[1].set_title('t-SNE — Latent Space (Pre-FDI)')
plt.tight_layout(); plt.show()


### Section 5: False Data Injection — Static (Additive / Multiplicative / Noise)

In [ ]:
def inject_static(df, injection_pct=0.05, seed=SEED):
    """
    Inject synthetic FDI into Gage, Warp62, Crosslevel.
    Uses numpy array operations — avoids pandas .loc indexing pitfalls.
    """
    rng        = np.random.default_rng(seed)
    n          = len(df)
    n_anomalies = int(n * injection_pct)
    window_len  = 46
    n_windows   = max(1, n_anomalies // window_len)
    spread      = (n - n_windows * window_len) // (n_windows + 1)

    arr   = df.values.copy().astype(float)   # numpy array — no .loc needed
    y     = np.zeros(n, dtype=int)
    cols  = list(df.columns)
    attack_col_idx = [cols.index(c) for c in ['Gage','Warp62','Crosslevel'] if c in cols]

    for i in range(n_windows):
        start = (i + 1) * spread + i * window_len
        end   = min(start + window_len, n)
        y[start:end] = 1

        method = rng.choice(['spike', 'multiply', 'noise'])
        for ci in attack_col_idx:
            std = arr[:, ci].std()
            if method == 'spike':
                arr[start:end, ci] += rng.choice([-1, 1]) * rng.uniform(2, 3) * std
            elif method == 'multiply':
                arr[start:end, ci] *= rng.uniform(1.5, 2.0)
            else:
                arr[start:end, ci] += rng.normal(0, 3 * std, size=(end - start))

    df_inj = pd.DataFrame(arr, columns=cols)
    print(f'[inject_static] pct={injection_pct*100:.0f}%  '
          f'windows={n_windows}  anomalies={int(y.sum())}/{n}')
    return df_inj, y

# ── Inject at 5% (main experiment) ──────────────────────────
df_inj5, y_true5 = inject_static(df_static, injection_pct=0.05)
X_inj5    = scaler_static.transform(df_inj5)
X_inj5_df = pd.DataFrame(X_inj5, columns=STATIC_COLS)


### Section 6: Post-Injection Detection — Static Models

In [ ]:
def run_static_detection(X_train, X_inj, y_true, enc, dec, iso, fw, label_prefix='5pct'):
    """
    Threshold calibrated on CLEAN TRAINING data only — no leakage.
    X_train : clean training samples
    X_inj   : test samples (with injected attacks)
    """
    train_vae = vae_reconstruction_error(enc, dec, X_train, fw)
    train_iso = -iso.score_samples(X_train)

    test_vae  = vae_reconstruction_error(enc, dec, X_inj, fw)
    test_iso  = -iso.score_samples(X_inj)

    results = {}
    for name, train_sc, test_sc in [
        ('IF',  train_iso, test_iso),
        ('VAE', train_vae, test_vae),
    ]:
        t      = train_percentile_threshold(train_sc, 95)
        y_pred = (test_sc > t).astype(int)
        results[f'{name}'] = evaluate(y_true, y_pred, f'{label_prefix} {name}')
    return results

static_results = {}
static_results['5pct'] = run_static_detection(
    X_static, X_inj5, y_true5, enc_s, dec_s, iso_s, fw_static, '5pct')


In [ ]:
# ── Confusion matrices for 5% injection — IF and VAE ────────────────
train_vae_err = vae_reconstruction_error(enc_s, dec_s, X_static, fw_static)
train_iso_sc  = -iso_s.score_samples(X_static)

test_vae_err  = vae_reconstruction_error(enc_s, dec_s, X_inj5, fw_static)
test_iso_sc   = -iso_s.score_samples(X_inj5)

for name, train_sc, test_sc in [
    ('IF',  train_iso_sc, test_iso_sc),
    ('VAE', train_vae_err, test_vae_err),
]:
    t = train_percentile_threshold(train_sc, 95)
    plot_confusion(y_true5, (test_sc > t).astype(int), f'{name} — 5% FDI')


In [ ]:
# ── PCA / t-SNE post-injection ───────────────────────────────
z_post = enc_s.predict(X_inj5, verbose=0)[0]
z_post_pca  = pca.transform(z_post[:3000])
z_post_tsne = TSNE(n_components=2, random_state=SEED, perplexity=30).fit_transform(z_post[:3000])
y_sub = y_true5[:3000]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, Z, title in zip(axes,
    [z_post_pca, z_post_tsne],
    ['PCA — Post-FDI (5%)', 't-SNE — Post-FDI (5%)']):
    ax.scatter(Z[y_sub==0, 0], Z[y_sub==0, 1], s=2, alpha=0.3, c='steelblue', label='Normal')
    ax.scatter(Z[y_sub==1, 0], Z[y_sub==1, 1], s=8, alpha=0.8, c='crimson', label='Injected')
    ax.set_title(title); ax.legend(markerscale=4, fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# ── Varying injection intensity: 2%, 5%, 10% ────────────────────────
injection_levels = [0.02, 0.05, 0.10]
all_static = {}

for pct in injection_levels:
    df_inj, y_inj = inject_static(df_static, injection_pct=pct)
    X_inj = scaler_static.transform(df_inj)
    all_static[pct] = run_static_detection(
        X_static, X_inj, y_inj, enc_s, dec_s, iso_s, fw_static,
        f'{int(pct*100)}pct')

# ── Summary plot ─────────────────────────────────────────────────────
models_  = ['IF', 'VAE']
metrics_ = ['f1', 'precision', 'recall']
x_labels = ['2%', '5%', '10%']

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, metric in zip(axes, metrics_):
    for m in models_:
        vals = [all_static[pct].get(m, {}).get(metric, 0) for pct in injection_levels]
        ax.plot(x_labels, vals, marker='o', label=m)
    ax.set_title(metric.upper()); ax.set_ylim(0, 1)
    ax.set_xlabel('Injection %'); ax.legend()
plt.suptitle('Static Detection — IF vs VAE across Injection Levels')
plt.tight_layout(); plt.show()


---
## PART 2 — TEMPORAL ANALYSIS (RawD.xlsx)
### Section 7: Load & Align 28-Month Inspection Data


In [ ]:
# ── Features shared with WRM_F ──────────────────────────────
TEMPORAL_FEATURES = ['GAGE', 'XLEVEL', 'WARP_62',
                     'SUF_L_62', 'SUF_R_62',
                     'ALI_L_62', 'ALI_R_62']
TEMPORAL_COLS_RENAME = {
    'GAGE': 'Gage', 'XLEVEL': 'Crosslevel', 'WARP_62': 'Warp62',
    'SUF_L_62': 'LProf62', 'SUF_R_62': 'RProf62',
    'ALI_L_62': 'LAlign62', 'ALI_R_62': 'RAlign62'
}

xl     = pd.ExcelFile(TEMPORAL_FILE)
sheets = [s for s in xl.sheet_names if s != 'Chart']
print(f'Inspection months: {len(sheets)}  ({sheets[0]} → {sheets[-1]})')

monthly_dfs = {}
for sheet in sheets:
    df = xl.parse(sheet)
    df = df[['MILE', 'FEET'] + TEMPORAL_FEATURES].dropna()
    df = df.rename(columns=TEMPORAL_COLS_RENAME)
    # ── KEY FIX: cast to int to avoid '64_1.0' vs '64_1' mismatch ──
    df['pos_key'] = (df['MILE'].astype(int).astype(str) + '_' +
                     df['FEET'].round(0).astype(int).astype(str))
    # Drop duplicate positions within same month (keep first)
    df = df.drop_duplicates('pos_key')
    monthly_dfs[sheet] = df.set_index('pos_key')

print('Sample shape per sheet:', next(iter(monthly_dfs.values())).shape)
# Sanity-check: show a few keys from first sheet
print('Sample keys:', sorted(monthly_dfs[sheets[0]].index)[:5])


In [ ]:
# ── Find positions present in ALL 28 months ─────────────────
common_keys = set(monthly_dfs[sheets[0]].index)
for s in sheets[1:]:
    common_keys &= set(monthly_dfs[s].index)
common_keys = sorted(common_keys)
print(f"Track positions in all months: {len(common_keys)}")

# ── Build tensor: (n_positions, n_months, n_features) ────────
FEAT_COLS = list(TEMPORAL_COLS_RENAME.values())
n_pos    = len(common_keys)
n_months = len(sheets)
n_feat   = len(FEAT_COLS)

X_temporal_raw = np.zeros((n_pos, n_months, n_feat))
for t, sheet in enumerate(sheets):
    df_t = monthly_dfs[sheet].loc[common_keys, FEAT_COLS]
    X_temporal_raw[:, t, :] = df_t.values

print(f"Temporal tensor shape: {X_temporal_raw.shape}")
print(f"  = ({n_pos} positions × {n_months} months × {n_feat} features)")


In [ ]:
# ── Preserve raw tensor — scaling happens AFTER train/test split ──────
# (Fitting scaler on all positions before split would leak test statistics
#  into the training normalisation.)
X_temporal = X_temporal_raw.copy()
print("Raw temporal tensor (unscaled):", X_temporal.shape)
print("Scaler will be fitted on training positions only in the next section.")

# ── Visualise raw signal for a few positions ──────────────────
fig, axes = plt.subplots(2, len(FEAT_COLS), figsize=(18, 5))
sample_positions = np.random.choice(n_pos, 10, replace=False)
months_label = [s.replace('_', '/') for s in sheets]
for j, feat in enumerate(FEAT_COLS):
    ax0, ax1 = axes[0, j], axes[1, j]
    for pos in sample_positions:
        ax0.plot(X_temporal[pos, :, j], alpha=0.4, lw=0.8)
    ax0.set_title(feat, fontsize=8); ax0.set_xticks([])
    ax1.plot(X_temporal[:, :, j].mean(axis=0), color='steelblue', lw=2)
    ax1.fill_between(range(n_months),
        X_temporal[:, :, j].mean(axis=0) - X_temporal[:, :, j].std(axis=0),
        X_temporal[:, :, j].mean(axis=0) + X_temporal[:, :, j].std(axis=0),
        alpha=0.2)
    ax1.set_xticks(range(0, n_months, 4))
    ax1.set_xticklabels(months_label[::4], rotation=45, fontsize=6)
axes[0, 0].set_ylabel('Sample pos.', fontsize=8)
axes[1, 0].set_ylabel('Mean ± std', fontsize=8)
plt.suptitle('Temporal Signals — 28-Month Track Geometry (raw, unscaled)', fontsize=12)
plt.tight_layout(); plt.show()


### Section 8: LSTM-VAE Architecture

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  LSTMVAEModel subclass — same Keras 3 pattern as VAEModel
# ─────────────────────────────────────────────────────────────────
class LSTMVAEModel(keras.Model):
    def __init__(self, encoder, decoder, **kwargs):
        super().__init__(**kwargs)
        self._enc = encoder
        self._dec = decoder


    def get_config(self):
        config = super().get_config()
        config.update({
            'encoder': keras.saving.serialize_keras_object(self._enc),
            'decoder': keras.saving.serialize_keras_object(self._dec),
        })
        return config

    @classmethod
    def from_config(cls, config):
        enc = keras.saving.deserialize_keras_object(config.pop('encoder'))
        dec = keras.saving.deserialize_keras_object(config.pop('decoder'))
        return cls(encoder=enc, decoder=dec, **config)
    def call(self, x, training=False):
        zm, zlv, z = self._enc(x, training=training)
        x_hat = self._dec(z, training=training)
        recon = kops.mean(
                    kops.mean(kops.sum(kops.square(x - x_hat), axis=-1), axis=-1))
        kl    = -0.5 * kops.mean(
                    kops.sum(1 + zlv - kops.square(zm) - kops.exp(zlv), axis=-1))
        self.add_loss(recon + kl)
        return x_hat

def build_lstm_vae(seq_len, n_features, latent_dim=16, lstm_units=64):
    # ── Encoder ──────────────────────────────────────────────
    enc_input = keras.Input(shape=(seq_len, n_features), name='lstm_enc_in')
    h = keras.layers.Bidirectional(
            keras.layers.LSTM(lstm_units, return_sequences=True))(enc_input)
    h = keras.layers.Dropout(0.1)(h)
    h = keras.layers.LSTM(lstm_units // 2)(h)
    h = keras.layers.BatchNormalization()(h)
    z_mean    = keras.layers.Dense(latent_dim, name='z_mean')(h)
    z_log_var = keras.layers.Dense(latent_dim, name='z_log_var')(h)
    z         = Sampling()([z_mean, z_log_var])
    encoder   = keras.Model(enc_input, [z_mean, z_log_var, z], name='lstm_encoder')

    # ── Decoder ──────────────────────────────────────────────
    dec_input  = keras.Input(shape=(latent_dim,), name='lstm_dec_in')
    d          = keras.layers.Dense(lstm_units // 2, activation='relu')(dec_input)
    d          = keras.layers.RepeatVector(seq_len)(d)
    d          = keras.layers.LSTM(lstm_units // 2, return_sequences=True)(d)
    d          = keras.layers.Dropout(0.1)(d)
    d          = keras.layers.LSTM(lstm_units, return_sequences=True)(d)
    dec_output = keras.layers.TimeDistributed(
                     keras.layers.Dense(n_features), name='recon')(d)
    decoder    = keras.Model(dec_input, dec_output, name='lstm_decoder')

    lstm_vae = LSTMVAEModel(encoder, decoder, name='lstm_vae')
    lstm_vae.compile(optimizer=keras.optimizers.Adam(1e-3))
    return lstm_vae, encoder, decoder

lstm_vae, lstm_enc, lstm_dec = build_lstm_vae(
    seq_len=n_months, n_features=n_feat, latent_dim=16, lstm_units=64)
lstm_vae.summary()


In [ ]:
# ── Split BEFORE scaling — prevents test-position leakage ───────────
from sklearn.model_selection import train_test_split

idx_all = np.arange(n_pos)
idx_train, idx_test = train_test_split(idx_all, test_size=0.2, random_state=SEED)

X_train_raw = X_temporal_raw[idx_train]
X_test_raw  = X_temporal_raw[idx_test]

# Fit scaler on training positions only, then transform both splits
scaler_t = StandardScaler()
X_train_t = scaler_t.fit_transform(
    X_train_raw.reshape(-1, n_feat)).reshape(X_train_raw.shape)
X_test_t  = scaler_t.transform(
    X_test_raw.reshape(-1, n_feat)).reshape(X_test_raw.shape)

print(f"Train: {X_train_t.shape}   Test: {X_test_t.shape}")
print("Scaler fitted on training positions only — no leakage ✓")

cb_lstm = [
    EarlyStopping(patience=15, restore_best_weights=True),
    ReduceLROnPlateau(patience=7, factor=0.5, verbose=0)
]
history_lstm = lstm_vae.fit(
    X_train_t, X_train_t,
    epochs=150, batch_size=128,
    validation_split=0.15,
    callbacks=cb_lstm, verbose=1
)

plt.figure(figsize=(7,3))
plt.plot(history_lstm.history['loss'], label='train')
plt.plot(history_lstm.history['val_loss'], label='val')
plt.xlabel('Epoch'); plt.ylabel('ELBO Loss'); plt.legend()
plt.title('LSTM-VAE — Training Curve'); plt.tight_layout(); plt.show()


### Section 8b: Transformer-VAE Architecture & Training


In [ ]:
# ─────────────────────────────────────────────────────────────────
#  Positional Encoding — sinusoidal, fixed (no training needed)
# ─────────────────────────────────────────────────────────────────
class PositionalEncoding(keras.layers.Layer):
    """Adds sinusoidal positional encoding to (batch, T, d_model) tensors."""
    def __init__(self, seq_len, d_model, **kwargs):
        super().__init__(**kwargs)
        self.seq_len = seq_len
        self.d_model = d_model
        pos   = np.arange(seq_len, dtype=np.float32)[:, np.newaxis]
        i     = np.arange(d_model, dtype=np.float32)[np.newaxis, :]
        angle = pos / np.power(10000.0, (2 * (i // 2)) / d_model)
        angle[:, 0::2] = np.sin(angle[:, 0::2])
        angle[:, 1::2] = np.cos(angle[:, 1::2])
        self._pe = tf.constant(angle[np.newaxis], dtype=tf.float32)  # (1,T,d)

    def call(self, x):
        return x + self._pe[:, :kops.shape(x)[1], :]

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'seq_len': self.seq_len, 'd_model': self.d_model})
        return cfg


# ─────────────────────────────────────────────────────────────────
#  Transformer-VAE wrapper  (same subclass pattern as LSTMVAEModel)
# ─────────────────────────────────────────────────────────────────
class TransformerVAEModel(keras.Model):
    """
    Transformer encoder + LSTM decoder VAE.
    Multi-head self-attention captures global temporal context so that
    any month inconsistent with the rest of the sequence raises loss —
    critical for one-shot and short persistent attacks.
    """
    def __init__(self, encoder, decoder, **kwargs):
        super().__init__(**kwargs)
        self._enc = encoder
        self._dec = decoder

    def get_config(self):
        cfg = super().get_config()
        cfg.update({
            'encoder': keras.saving.serialize_keras_object(self._enc),
            'decoder': keras.saving.serialize_keras_object(self._dec),
        })
        return cfg

    @classmethod
    def from_config(cls, config):
        enc = keras.saving.deserialize_keras_object(config.pop('encoder'))
        dec = keras.saving.deserialize_keras_object(config.pop('decoder'))
        return cls(encoder=enc, decoder=dec, **config)

    def call(self, x, training=False):
        zm, zlv, z = self._enc(x, training=training)
        x_hat = self._dec(z, training=training)
        recon = kops.mean(
                    kops.mean(kops.sum(kops.square(x - x_hat), axis=-1), axis=-1))
        kl    = -0.5 * kops.mean(
                    kops.sum(1 + zlv - kops.square(zm) - kops.exp(zlv), axis=-1))
        self.add_loss(recon + kl)
        return x_hat


def build_transformer_vae(seq_len, n_features, latent_dim=16,
                           d_model=64, n_heads=4, n_layers=2, dropout=0.1):
    """
    Encoder : Dense projection → Positional Encoding
              → N × (MultiHeadAttention + LayerNorm + FFN + LayerNorm)
              → GlobalAveragePooling → BatchNorm → z_mean / z_log_var
    Decoder : Dense → RepeatVector → LSTM × 2 → TimeDistributed Dense
    """
    # ── Encoder ──────────────────────────────────────────────
    enc_input = keras.Input(shape=(seq_len, n_features), name='trans_enc_in')
    x  = keras.layers.Dense(d_model)(enc_input)          # project features → d_model
    x  = PositionalEncoding(seq_len, d_model)(x)         # add positional context

    for _ in range(n_layers):
        attn = keras.layers.MultiHeadAttention(
                   num_heads=n_heads, key_dim=d_model // n_heads,
                   dropout=dropout)(x, x)                # self-attention
        x    = keras.layers.LayerNormalization(epsilon=1e-6)(x + attn)
        ff   = keras.layers.Dense(d_model * 2, activation='relu')(x)
        ff   = keras.layers.Dropout(dropout)(ff)
        ff   = keras.layers.Dense(d_model)(ff)
        x    = keras.layers.LayerNormalization(epsilon=1e-6)(x + ff)

    h         = keras.layers.GlobalAveragePooling1D()(x)  # pool over time
    h         = keras.layers.BatchNormalization()(h)
    z_mean    = keras.layers.Dense(latent_dim, name='z_mean')(h)
    z_log_var = keras.layers.Dense(latent_dim, name='z_log_var')(h)
    z         = Sampling()([z_mean, z_log_var])
    encoder   = keras.Model(enc_input, [z_mean, z_log_var, z], name='trans_encoder')

    # ── Decoder (same as LSTM-VAE) ────────────────────────────
    dec_input  = keras.Input(shape=(latent_dim,), name='trans_dec_in')
    d          = keras.layers.Dense(d_model // 2, activation='relu')(dec_input)
    d          = keras.layers.RepeatVector(seq_len)(d)
    d          = keras.layers.LSTM(d_model // 2, return_sequences=True)(d)
    d          = keras.layers.Dropout(dropout)(d)
    d          = keras.layers.LSTM(d_model, return_sequences=True)(d)
    dec_output = keras.layers.TimeDistributed(
                     keras.layers.Dense(n_features))(d)
    decoder    = keras.Model(dec_input, dec_output, name='trans_decoder')

    trans_vae = TransformerVAEModel(encoder, decoder, name='trans_vae')
    trans_vae.compile(optimizer=keras.optimizers.Adam(1e-3))
    return trans_vae, encoder, decoder


trans_vae, trans_enc, trans_dec = build_transformer_vae(
    seq_len=n_months, n_features=n_feat,
    latent_dim=16, d_model=64, n_heads=4, n_layers=2)
trans_vae.summary()


In [ ]:
# ── Train Transformer-VAE on clean sequences ────────────────
print('Training Transformer-VAE...')
cb_trans = [
    EarlyStopping(patience=15, restore_best_weights=True),
    ReduceLROnPlateau(patience=7, factor=0.5, verbose=0)
]
history_trans = trans_vae.fit(
    X_train_t, X_train_t,
    epochs=150, batch_size=128,
    validation_split=0.15,
    callbacks=cb_trans, verbose=1
)

plt.figure(figsize=(7,3))
plt.plot(history_trans.history['loss'],     label='train')
plt.plot(history_trans.history['val_loss'], label='val')
plt.xlabel('Epoch'); plt.ylabel('ELBO Loss'); plt.legend()
plt.title('Transformer-VAE — Training Curve')
plt.tight_layout(); plt.show()


### Section 9: Temporal FDI Injection — Persistent & One-Shot

In [ ]:
def inject_temporal(X_seq, injection_pct=0.05, attack_type='persistent',
                    persist_months=6, seed=SEED):
    """
    Inject FDI into temporal sequences.
    attack_type='persistent' : tamper `persist_months` consecutive months
    attack_type='one_shot'   : tamper a single random month
    Returns (X_injected, y_seq_labels, y_step_labels)
    """
    rng      = np.random.default_rng(seed)
    n_seq, n_t, n_f = X_seq.shape
    n_attack = max(1, int(n_seq * injection_pct))

    # Guard: persist_months must be < n_t
    persist_months = min(persist_months, n_t - 1)

    X_inj      = X_seq.copy()
    y_seq      = np.zeros(n_seq, dtype=int)
    y_step     = np.zeros((n_seq, n_t), dtype=int)
    attack_idx = rng.choice(n_seq, size=n_attack, replace=False)
    y_seq[attack_idx] = 1

    feat_idx = [0, 1, 2]   # Gage, Crosslevel, Warp62

    for idx in attack_idx:
        if attack_type == 'persistent':
            max_start = n_t - persist_months
            t_start   = int(rng.integers(0, max(1, max_start)))
            t_end     = t_start + persist_months
            y_step[idx, t_start:t_end] = 1
            method = rng.choice(['spike', 'drift', 'noise'])
            for f in feat_idx:
                if method == 'spike':
                    amp = float(rng.choice([-1, 1])) * rng.uniform(2, 3)
                    X_inj[idx, t_start:t_end, f] += amp
                elif method == 'drift':
                    drift = np.linspace(0, rng.uniform(1.5, 3.0), persist_months)
                    X_inj[idx, t_start:t_end, f] += drift
                else:
                    X_inj[idx, t_start:t_end, f] += rng.normal(0, 2.5, persist_months)
        else:   # one_shot
            t_atk = int(rng.integers(0, n_t))
            y_step[idx, t_atk] = 1
            method = rng.choice(['spike', 'noise'])
            for f in feat_idx:
                if method == 'spike':
                    X_inj[idx, t_atk, f] += float(rng.choice([-1, 1])) * rng.uniform(3, 5)
                else:
                    X_inj[idx, t_atk, f] += rng.normal(0, 4)

    print(f'[inject_temporal] type={attack_type}  pct={injection_pct*100:.0f}%  '
          f'attacked={int(y_seq.sum())}/{n_seq}')
    return X_inj, y_seq, y_step

# ── Preview all injection configs ───────────────────────────
for atk_type in ['persistent', 'one_shot']:
    for pct in [0.02, 0.05, 0.10]:
        Xi, yi, _ = inject_temporal(X_test_t, injection_pct=pct, attack_type=atk_type)
        print(f'  {atk_type} {int(pct*100)}%: shape={Xi.shape}  attacks={yi.sum()}')


### Section 10: LSTM-VAE Anomaly Detection

In [ ]:
def lstm_reconstruction_error(encoder, decoder, X_seq, reduction='max'):
    """
    Per-sequence reconstruction error.
    reduction='max'  — max over timesteps (default).
                       Prevents dilution of localised attacks across 28 months.
    reduction='mean' — mean over timesteps.
    """
    _, _, z = encoder.predict(X_seq, verbose=0)
    X_hat   = decoder.predict(z, verbose=0)
    per_step = np.mean((X_seq - X_hat)**2, axis=-1)   # (N, T) mean over features
    if reduction == 'max':
        return np.max(per_step, axis=-1)               # (N,)  max over timesteps
    return np.mean(per_step, axis=-1)

def run_temporal_detection(X_train, X_test_inj, y_seq_true,
                            enc, dec, label='LSTM-VAE'):
    """
    Threshold from TRAINING distribution only — no label leakage.
    MAD calibrated on training reconstruction errors.
    """
    train_err = lstm_reconstruction_error(enc, dec, X_train)
    test_err  = lstm_reconstruction_error(enc, dec, X_test_inj)
    test_mad  = mad_score_from_train(train_err, test_err)
    train_mad = mad_score_from_train(train_err, train_err)

    results = {}
    for name, train_sc, test_sc in [
        (label,          train_err, test_err),
        (label + '+MAD', train_mad, test_mad),
    ]:
        t      = train_percentile_threshold(train_sc, 95)
        y_pred = (test_sc > t).astype(int)
        results[name] = evaluate(y_seq_true, y_pred, name)
    return results, test_err, test_mad

# ── Rerun with clean thresholds ──────────────────────────────
X_inj_p5, y_p5, _ = inject_temporal(X_test_t, 0.05, 'persistent')
res_p5, err_p5, mad_p5 = run_temporal_detection(
    X_train_t, X_inj_p5, y_p5, lstm_enc, lstm_dec, 'LSTM-VAE (Persistent-5%)')

X_inj_o5, y_o5, _ = inject_temporal(X_test_t, 0.05, 'one_shot')
res_o5, err_o5, mad_o5 = run_temporal_detection(
    X_train_t, X_inj_o5, y_o5, lstm_enc, lstm_dec, 'LSTM-VAE (OneShot-5%)')


In [ ]:
# ── Reconstruction error distributions ──────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, err, y_lbl, title in zip(axes,
    [err_p5, err_o5],
    [y_p5,   y_o5],
    ['Persistent Attack (5%)', 'One-Shot Attack (5%)']):
    ax.hist(err[y_lbl==0], bins=50, alpha=0.6, label='Normal', color='steelblue')
    ax.hist(err[y_lbl==1], bins=50, alpha=0.6, label='Injected', color='crimson')
    ax.set_title(f'LSTM-VAE Recon Error — {title}')
    ax.set_xlabel('Reconstruction Error'); ax.legend()
plt.tight_layout(); plt.show()

# ── Latent space post-injection ──────────────────────────────
z_inj_p = lstm_enc.predict(X_inj_p5, verbose=0)[0]
pca_t   = PCA(n_components=2).fit_transform(z_inj_p)
plt.figure(figsize=(6,4))
plt.scatter(pca_t[y_p5==0,0], pca_t[y_p5==0,1], s=4, alpha=0.3, label='Normal')
plt.scatter(pca_t[y_p5==1,0], pca_t[y_p5==1,1], s=15, alpha=0.8, c='crimson', label='Injected')
plt.title('LSTM-VAE Latent Space — Persistent FDI (5%)')
plt.legend(markerscale=3); plt.tight_layout(); plt.show()


In [ ]:
temporal_results = {}
for atk_type in ['persistent', 'one_shot']:
    temporal_results[atk_type] = {}
    for pct in [0.02, 0.05, 0.10]:
        X_inj, y_inj, _ = inject_temporal(X_test_t, pct, atk_type)
        res, _, _ = run_temporal_detection(
            X_train_t, X_inj, y_inj, lstm_enc, lstm_dec,
            f'LSTM-VAE ({atk_type}-{int(pct*100)}%)')
        temporal_results[atk_type][pct] = res


In [ ]:
# ── Trans-VAE detection — 5% attacks ───────────────────────
X_inj_tp5, y_tp5, _ = inject_temporal(X_test_t, 0.05, 'persistent')
res_tp5, err_tp5, mad_tp5 = run_temporal_detection(
    X_train_t, X_inj_tp5, y_tp5, trans_enc, trans_dec,
    'Trans-VAE (Persistent-5%)')

X_inj_to5, y_to5, _ = inject_temporal(X_test_t, 0.05, 'one_shot')
res_to5, err_to5, mad_to5 = run_temporal_detection(
    X_train_t, X_inj_to5, y_to5, trans_enc, trans_dec,
    'Trans-VAE (OneShot-5%)')

# ── Reconstruction error distribution comparison ─────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, err_l, err_t, y_lbl, title in zip(axes,
    [err_p5,    err_o5],
    [err_tp5,   err_to5],
    [y_p5,      y_o5],
    ['Persistent 5%', 'One-Shot 5%']):
    ax.hist(err_l[y_lbl==0], bins=40, alpha=0.4, color='steelblue', label='Normal (LSTM-VAE)')
    ax.hist(err_l[y_lbl==1], bins=40, alpha=0.4, color='crimson',   label='Attacked (LSTM-VAE)')
    ax.hist(err_t[y_lbl==0], bins=40, alpha=0.4, color='green',     label='Normal (Trans-VAE)')
    ax.hist(err_t[y_lbl==1], bins=40, alpha=0.4, color='orange',    label='Attacked (Trans-VAE)')
    ax.set_title(f'Recon Error — {title}')
    ax.set_xlabel('Reconstruction Error'); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

# ── Sweep 2 / 5 / 10% ────────────────────────────────────────
trans_results = {}
for atk_type in ['persistent', 'one_shot']:
    trans_results[atk_type] = {}
    for pct in [0.02, 0.05, 0.10]:
        X_inj, y_inj, _ = inject_temporal(X_test_t, pct, atk_type)
        res, _, _ = run_temporal_detection(
            X_train_t, X_inj, y_inj, trans_enc, trans_dec,
            f'Trans-VAE ({atk_type}-{int(pct*100)}%)')
        trans_results[atk_type][pct] = res


---
## PART 3 — STATIC vs TEMPORAL COMPARISON
### Section 11: Results Summary & Statistical Testing


In [ ]:
rows = []
for pct in [0.02, 0.05, 0.10]:
    pct_label = f'{int(pct*100)}%'
    # Static models (WRM_F)
    for m in ['IF', 'VAE']:
        r = all_static[pct][m]
        rows.append({'Framework': 'Static', 'Model': m,
                     'Attack Type': 'Spike/Mult/Noise', 'FDI %': pct_label,
                     'Precision': r['precision'], 'Recall': r['recall'],
                     'F1': r['f1'], 'Accuracy': r['accuracy']})
    # Temporal models (RawD)
    for atk in ['persistent', 'one_shot']:
        res_dict = temporal_results[atk][pct]
        for m_key in ['LSTM-VAE']:
            matched = [k for k in res_dict if 'LSTM-VAE' in k and '+MAD' not in k]
            if matched:
                r = res_dict[matched[0]]
                rows.append({'Framework': 'Temporal', 'Model': m_key,
                             'Attack Type': atk.replace('_', '-'), 'FDI %': pct_label,
                             'Precision': r['precision'], 'Recall': r['recall'],
                             'F1': r['f1'], 'Accuracy': r['accuracy']})
    for atk in ['persistent', 'one_shot']:
        res_dict = trans_results[atk][pct]
        matched = [k for k in res_dict if 'Trans-VAE' in k and '+MAD' not in k]
        if matched:
            r = res_dict[matched[0]]
            rows.append({'Framework': 'Temporal', 'Model': 'Trans-VAE',
                         'Attack Type': atk.replace('_', '-'), 'FDI %': pct_label,
                         'Precision': r['precision'], 'Recall': r['recall'],
                         'F1': r['f1'], 'Accuracy': r['accuracy']})

summary_df = pd.DataFrame(rows)
pd.set_option('display.float_format', '{:.3f}'.format)
print(summary_df.to_string(index=False))


In [ ]:
# ── F1 bar chart across injection levels ────────────────────
injection_pcts = [0.02, 0.05, 0.10]
bar_w = 0.2

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, pct, title in zip(axes, injection_pcts, ['FDI 2%', 'FDI 5%', 'FDI 10%']):
    model_names = ['IF', 'VAE', 'LSTM-VAE', 'Trans-VAE']
    xs = np.arange(len(model_names))

    def get_f1_temporal(res_dict, pct, substr):
        r = res_dict.get('persistent', {}).get(pct, {})
        key = next((k for k in r if substr in k and '+MAD' not in k), None)
        return r[key]['f1'] if key else 0.0

    def get_f1_oneshot(res_dict, pct, substr):
        r = res_dict.get('one_shot', {}).get(pct, {})
        key = next((k for k in r if substr in k and '+MAD' not in k), None)
        return r[key]['f1'] if key else 0.0

    f1_static  = [all_static[pct].get('IF',  {}).get('f1', 0),
                  all_static[pct].get('VAE', {}).get('f1', 0), 0, 0]
    f1_persist = [0, 0,
                  get_f1_temporal(temporal_results, pct, 'LSTM-VAE'),
                  get_f1_temporal(trans_results,    pct, 'Trans-VAE')]
    f1_oneshot = [0, 0,
                  get_f1_oneshot(temporal_results, pct, 'LSTM-VAE'),
                  get_f1_oneshot(trans_results,    pct, 'Trans-VAE')]

    ax.bar(xs - bar_w, f1_static,  bar_w, label='Spike/Mult/Noise', color='#4472C4')
    ax.bar(xs,          f1_persist, bar_w, label='Persistent',       color='#ED7D31')
    ax.bar(xs + bar_w,  f1_oneshot, bar_w, label='One-Shot',         color='#FF0000')

    ax.set_xticks(xs); ax.set_xticklabels(model_names, fontsize=9)
    ax.set_ylim(0, 1.05); ax.set_ylabel('F1-Score'); ax.set_title(title)
    ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)

plt.suptitle('F1-Score: Static (WRM_F) vs Temporal (RawD) Models', fontsize=13)
plt.tight_layout(); plt.show()


In [ ]:
# ── Grouped bar: F1 comparison across models & attack types ─
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
for ax, pct in zip(axes, ['2%', '5%', '10%']):
    sub = summary_df[summary_df['FDI %'] == pct]
    models_u = sub['Model'].unique()
    x = np.arange(len(models_u))
    width = 0.25
    atk_colors = {'Spike/Mult/Noise':'steelblue', 'persistent':'darkorange', 'one-shot':'crimson'}
    for i, atk in enumerate(sub['Attack Type'].unique()):
        vals = sub[sub['Attack Type'] == atk].set_index('Model')['F1']
        vals = [vals.get(m, 0) for m in models_u]
        ax.bar(x + i*width, vals, width, label=atk,
               color=list(atk_colors.values())[i % 3], alpha=0.8)
    ax.set_xticks(x + width)
    ax.set_xticklabels(models_u, rotation=25, fontsize=8)
    ax.set_title(f'FDI {pct}'); ax.set_ylabel('F1-Score')
    ax.set_ylim(0, 1.05); ax.legend(fontsize=7); ax.grid(axis='y', alpha=0.3)
plt.suptitle('F1-Score: Static vs Temporal Models', fontsize=13)
plt.tight_layout(); plt.show()


In [ ]:
# McNemar test removed.
# Static and temporal models operate on different datasets (WRM_F vs RawD)
# and cannot be directly compared via McNemar's test.
# Cross-model comparison is handled via ROC-AUC in Section 11.
pass


In [ ]:
# ── Radar chart: all models at 5% ───────────────────────────
from matplotlib.patches import FancyArrowPatch
import matplotlib.patches as mpatches

metrics_radar = ['Precision', 'Recall', 'F1', 'Accuracy']
model_data = summary_df[summary_df['FDI %'] == '5%'].groupby('Model')[metrics_radar].mean()

angles = np.linspace(0, 2*np.pi, len(metrics_radar), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
colors_ = plt.cm.tab10(np.linspace(0, 1, len(model_data)))

for (model_name, row), col in zip(model_data.iterrows(), colors_):
    vals = row.tolist() + row.tolist()[:1]
    ax.plot(angles, vals, '-', lw=2, color=col, label=model_name)
    ax.fill(angles, vals, alpha=0.1, color=col)

ax.set_thetagrids(np.degrees(angles[:-1]), metrics_radar)
ax.set_ylim(0, 1)
ax.set_title('Model Performance Radar — 5% FDI', pad=20, fontsize=13)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1))
plt.tight_layout(); plt.show()


In [ ]:
# ── ROC — all models at 5% FDI ─────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))

# Static VAE (raw reconstruction error)
vae_s_roc = vae_reconstruction_error(enc_s, dec_s, X_inj5, fw_static)
fpr, tpr, _ = roc_curve(y_true5, vae_s_roc)
ax.plot(fpr, tpr, lw=2, color='steelblue',
        label=f'Static VAE (AUC={roc_auc_score(y_true5, vae_s_roc):.3f})')

# Static IF
if_sc = -iso_s.score_samples(X_inj5)
fpr2, tpr2, _ = roc_curve(y_true5, if_sc)
ax.plot(fpr2, tpr2, lw=2, ls='--', color='gray',
        label=f'Static IF (AUC={roc_auc_score(y_true5, if_sc):.3f})')

# LSTM-VAE persistent
fpr3, tpr3, _ = roc_curve(y_p5, err_p5)
ax.plot(fpr3, tpr3, lw=2, color='crimson',
        label=f'LSTM-VAE Persistent (AUC={roc_auc_score(y_p5, err_p5):.3f})')

# LSTM-VAE one-shot
fpr4, tpr4, _ = roc_curve(y_o5, err_o5)
ax.plot(fpr4, tpr4, lw=2, ls='-.', color='darkorange',
        label=f'LSTM-VAE One-Shot (AUC={roc_auc_score(y_o5, err_o5):.3f})')

# Trans-VAE persistent
fpr5, tpr5, _ = roc_curve(y_tp5, err_tp5)
ax.plot(fpr5, tpr5, lw=2, ls=':', color='darkgreen',
        label=f'Trans-VAE Persistent (AUC={roc_auc_score(y_tp5, err_tp5):.3f})')

# Trans-VAE one-shot
fpr6, tpr6, _ = roc_curve(y_to5, err_to5)
ax.plot(fpr6, tpr6, lw=2, ls=(0,(3,1,1,1)), color='purple',
        label=f'Trans-VAE One-Shot (AUC={roc_auc_score(y_to5, err_to5):.3f})')

ax.plot([0,1],[0,1],'k--', alpha=0.4)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — 5% FDI Intensity')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# ── Bootstrap 95% Confidence Intervals ─────────────────────────────────
from sklearn.utils import resample

def bootstrap_auc_f1(y_true, scores, threshold, n_boot=2000, seed=SEED):
    """Bootstrap CI for AUC and F1 at a fixed operating threshold."""
    rng = np.random.RandomState(seed)
    aucs, f1s = [], []
    for _ in range(n_boot):
        idx = resample(np.arange(len(y_true)), random_state=rng)
        yt, ys = y_true[idx], scores[idx]
        if len(np.unique(yt)) < 2:
            continue
        aucs.append(roc_auc_score(yt, ys))
        f1s.append(f1_score(yt, (ys > threshold).astype(int), zero_division=0))
    a = 0.025
    return {
        'auc_mean': np.mean(aucs),
        'auc_lo':   np.percentile(aucs, 100*a),
        'auc_hi':   np.percentile(aucs, 100*(1-a)),
        'f1_mean':  np.mean(f1s),
        'f1_lo':    np.percentile(f1s,  100*a),
        'f1_hi':    np.percentile(f1s,  100*(1-a)),
    }

# Training thresholds (no leakage)
_train_vae_err  = vae_reconstruction_error(enc_s, dec_s, X_static, fw_static)
_train_lstm_err = lstm_reconstruction_error(lstm_enc, lstm_dec, X_train_t)
_train_trans_err = lstm_reconstruction_error(trans_enc, trans_dec, X_train_t)

thresh_vae   = train_percentile_threshold(_train_vae_err)
thresh_lstm  = train_percentile_threshold(_train_lstm_err)
thresh_trans = train_percentile_threshold(_train_trans_err)

_test_vae5 = vae_reconstruction_error(enc_s, dec_s, X_inj5, fw_static)

print(f"{'Model / Attack':<40}  {'AUC':<5}  {'95% CI AUC':<15}  {'F1':<5}  {'95% CI F1'}")
print("-" * 87)

ci_results = {}
for label, yt, sc, thr in [
    ('VAE — Spike/Mult/Noise 5%',   y_true5, _test_vae5, thresh_vae),
    ('LSTM-VAE — Persistent 5%',    y_p5,    err_p5,     thresh_lstm),
    ('Trans-VAE — Persistent 5%',   y_tp5,   err_tp5,    thresh_trans),
    ('LSTM-VAE — One-Shot 5%',      y_o5,    err_o5,     thresh_lstm),
    ('Trans-VAE — One-Shot 5%',     y_to5,   err_to5,    thresh_trans),
]:
    ci = bootstrap_auc_f1(yt, sc, thr)
    ci_results[label] = ci
    print(f"{label:<40}  {ci['auc_mean']:.3f}  "
          f"[{ci['auc_lo']:.3f}–{ci['auc_hi']:.3f}]  "
          f"{ci['f1_mean']:.3f}  [{ci['f1_lo']:.3f}–{ci['f1_hi']:.3f}]")


### Section 12: Save Models & Results

In [ ]:
import os

SAVE_DIR = BASE_PATH + 'FDI_Results/'
os.makedirs(SAVE_DIR, exist_ok=True)

# Save models
vae_s.save(SAVE_DIR + 'static_vae.keras')
lstm_vae.save(SAVE_DIR + 'lstm_vae.keras')
print("Models saved.")

# Save summary table
summary_df.to_csv(SAVE_DIR + 'results_summary.csv', index=False)
print("Results CSV saved.")

# Save temporal scaler
import pickle
with open(SAVE_DIR + 'scaler_temporal.pkl', 'wb') as f:
    pickle.dump(scaler_t, f)
with open(SAVE_DIR + 'scaler_static.pkl', 'wb') as f:
    pickle.dump(scaler_static, f)
print("Scalers saved.")
print(f"\nAll outputs saved to: {SAVE_DIR}")


In [ ]:
# ── Layer 2: CUSUM for Persistent Drift (T0856) ────────────────────────

def fit_cusum(X_train_seq):
    X_flat = X_train_seq.reshape(-1, X_train_seq.shape[-1])
    mu0    = X_flat.mean(axis=0)
    sigma  = X_flat.std(axis=0) + 1e-8
    return mu0, sigma

ATTACK_FEAT_IDX = [0, 1, 2]   # Gage, Crosslevel, Warp62 — features injected into

def cusum_score(X_seq, mu0, sigma, k_factor=0.5, feat_idx=ATTACK_FEAT_IDX):
    # Restrict to attacked features only — avoids noise dilution from clean features
    if feat_idx is not None:
        X_seq  = X_seq[:, :, feat_idx]
        mu0    = mu0[feat_idx]
        sigma  = sigma[feat_idx]
    scores = np.zeros(len(X_seq))
    for i, seq in enumerate(X_seq):
        S_pos = np.zeros(seq.shape[1])
        S_neg = np.zeros(seq.shape[1])
        peak  = 0.0
        for t in range(len(seq)):
            dev   = (seq[t] - mu0) / sigma
            S_pos = np.maximum(0.0, S_pos + dev - k_factor)
            S_neg = np.maximum(0.0, S_neg - dev - k_factor)
            peak  = max(peak, S_pos.max(), S_neg.max())
        scores[i] = peak
    return scores

# Fit on training data
mu0_cu, sigma_cu   = fit_cusum(X_train_t)
train_cu           = cusum_score(X_train_t, mu0_cu, sigma_cu)
thresh_cu          = np.percentile(train_cu, 95)

# Evaluate on persistent 5% test set (already created in cell 38)
test_cu_p5         = cusum_score(X_inj_p5, mu0_cu, sigma_cu)
pred_cu            = (test_cu_p5 > thresh_cu).astype(int)

auc_cu  = roc_auc_score(y_p5, test_cu_p5)
f1_cu   = f1_score(y_p5, pred_cu)
print(f"CUSUM  |  AUC={auc_cu:.3f}   F1={f1_cu:.3f}   threshold={thresh_cu:.3f}")

# ROC comparison: CUSUM vs LSTM-VAE vs Trans-VAE on persistent drift
fig, ax = plt.subplots(figsize=(7, 5))
for scores, y_lbl, label, color, ls in [
    (err_p5,    y_p5,  'LSTM-VAE',  'crimson',   '--'),
    (err_tp5,   y_tp5, 'Trans-VAE', 'darkgreen',  ':'),
    (test_cu_p5, y_p5, 'CUSUM (proposed)', 'steelblue', '-'),
]:
    fpr, tpr, _ = roc_curve(y_lbl, scores)
    auc = roc_auc_score(y_lbl, scores)
    ax.plot(fpr, tpr, lw=2, ls=ls, color=color, label=f'{label} (AUC={auc:.3f})')

ax.plot([0,1],[0,1],'k--', alpha=0.3, label='Random')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('Persistent Drift Detection (5% FDI)\nLayer 2: CUSUM vs Reconstruction-Based')
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ── Layer 3: Stagnation Score for Replay (T0856) ───────────────────────

# Re-define inject_temporal_replay in case cell 36 was skipped
def inject_temporal_replay(X_seq, pct, replay_window=6, seed=SEED):
    rng      = np.random.default_rng(seed)
    n_seq, n_t, n_f = X_seq.shape
    n_attack = max(1, int(n_seq * pct))
    attack_indices = rng.choice(n_seq, n_attack, replace=False)
    X_inj = X_seq.copy()
    y     = np.zeros(n_seq, dtype=int)
    for idx in attack_indices:
        y[idx] = 1
        src_start = int(rng.integers(0, n_t // 2 - replay_window + 1))
        tgt_start = n_t - replay_window
        X_inj[idx, tgt_start:, :] = X_seq[idx, src_start:src_start + replay_window, :]
    return X_inj, y, attack_indices

def stagnation_score(X_seq, window=6):
    N, T, F = X_seq.shape
    scores  = np.zeros(N)
    for i, seq in enumerate(X_seq):
        last_win = seq[T - window:]
        min_mse  = np.inf
        for s in range(0, T - 2 * window + 1):
            mse = np.mean((last_win - seq[s : s + window]) ** 2)
            if mse < min_mse:
                min_mse = mse
        scores[i] = min_mse
    return scores

# Fresh replay injection at 5%
X_inj_r5, y_r5, _ = inject_temporal_replay(X_test_t, 0.05)

train_stag  = stagnation_score(X_train_t, window=6)
thresh_stag = np.percentile(train_stag, 5)

test_stag   = stagnation_score(X_inj_r5, window=6)
pred_stag   = (test_stag < thresh_stag).astype(int)

err_r5_lstm  = lstm_reconstruction_error(lstm_enc,  lstm_dec,  X_inj_r5)
err_r5_trans = lstm_reconstruction_error(trans_enc, trans_dec, X_inj_r5)

auc_stag = roc_auc_score(y_r5, -test_stag)
f1_stag  = f1_score(y_r5, pred_stag)
print(f"Stagnation | AUC={auc_stag:.3f}   F1={f1_stag:.3f}")

fig, ax = plt.subplots(figsize=(7, 5))
for scores, y_lbl, label, color, ls in [
    (err_r5_lstm,  y_r5, 'LSTM-VAE',             'crimson',   '--'),
    (err_r5_trans, y_r5, 'Trans-VAE',             'darkgreen',  ':'),
    (-test_stag,   y_r5, 'Stagnation (proposed)', 'steelblue',  '-'),
]:
    fpr, tpr, _ = roc_curve(y_lbl, scores)
    auc = roc_auc_score(y_lbl, scores)
    ax.plot(fpr, tpr, lw=2, ls=ls, color=color, label=f'{label} (AUC={auc:.3f})')

ax.plot([0,1],[0,1],'k--', alpha=0.3, label='Random')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('Replay Attack Detection (5% FDI)\nLayer 3: Stagnation Score vs Reconstruction-Based')
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ── Adversarially Hardened Replay Variants ─────────────────────────────

import numpy as np
from sklearn.metrics import roc_auc_score

ATTACK_FEAT_IDX = [0, 1, 2]  # Gage, Crosslevel, Warp62

def inject_replay_noisy(X_seq, pct, noise_sigma=0.05, replay_window=6, seed=SEED):
    """Replay + Gaussian noise: xt = x_{t-k} + ε,  ε ~ N(0, noise_sigma²).
    noise_sigma is expressed as fraction of per-feature std from training."""
    rng = np.random.default_rng(seed)
    feat_std = X_seq.reshape(-1, X_seq.shape[-1]).std(axis=0)
    n_seq, n_t, n_f = X_seq.shape
    n_attack = max(1, int(n_seq * pct))
    attack_idx = rng.choice(n_seq, n_attack, replace=False)
    X_inj = X_seq.copy()
    y = np.zeros(n_seq, dtype=int)
    for idx in attack_idx:
        y[idx] = 1
        src_start = int(rng.integers(0, n_t // 2 - replay_window + 1))
        tgt_start = n_t - replay_window
        replayed = X_seq[idx, src_start:src_start + replay_window, :].copy()
        noise = rng.normal(0, noise_sigma * feat_std, replayed.shape)
        X_inj[idx, tgt_start:, :] = replayed + noise
    return X_inj, y

def inject_replay_drift(X_seq, pct, drift_rate=0.05, replay_window=6, seed=SEED):
    """Replay + linear drift: xt = x_{t-k} + δ·t.
    drift_rate is fraction of per-feature std per month."""
    rng = np.random.default_rng(seed)
    feat_std = X_seq.reshape(-1, X_seq.shape[-1]).std(axis=0)
    n_seq, n_t, n_f = X_seq.shape
    n_attack = max(1, int(n_seq * pct))
    attack_idx = rng.choice(n_seq, n_attack, replace=False)
    X_inj = X_seq.copy()
    y = np.zeros(n_seq, dtype=int)
    for idx in attack_idx:
        y[idx] = 1
        src_start = int(rng.integers(0, n_t // 2 - replay_window + 1))
        tgt_start = n_t - replay_window
        replayed = X_seq[idx, src_start:src_start + replay_window, :].copy()
        for t in range(replay_window):
            replayed[t] += drift_rate * feat_std * t
        X_inj[idx, tgt_start:, :] = replayed
    return X_inj, y

def inject_replay_partial(X_seq, pct, feat_idx=None, replay_window=6, seed=SEED):
    """Replay only selected features — attacker only spoofs Gage sensor."""
    if feat_idx is None:
        feat_idx = [0]   # Gage only
    rng = np.random.default_rng(seed)
    n_seq, n_t, n_f = X_seq.shape
    n_attack = max(1, int(n_seq * pct))
    attack_idx = rng.choice(n_seq, n_attack, replace=False)
    X_inj = X_seq.copy()
    y = np.zeros(n_seq, dtype=int)
    for idx in attack_idx:
        y[idx] = 1
        src_start = int(rng.integers(0, n_t // 2 - replay_window + 1))
        tgt_start = n_t - replay_window
        X_inj[idx, tgt_start:, feat_idx] = X_seq[idx, src_start:src_start + replay_window, feat_idx]
    return X_inj, y

# ── Run all variants at 5% FDI ──────────────────────────────────────────
variants = {
    'Exact replay':       inject_temporal_replay(X_test_t, 0.05),
    'Replay+noise σ=0.05': inject_replay_noisy(X_test_t, 0.05, noise_sigma=0.05),
    'Replay+noise σ=0.10': inject_replay_noisy(X_test_t, 0.05, noise_sigma=0.10),
    'Replay+noise σ=0.20': inject_replay_noisy(X_test_t, 0.05, noise_sigma=0.20),
    'Replay+drift δ=0.05': inject_replay_drift(X_test_t, 0.05, drift_rate=0.05),
    'Replay+drift δ=0.10': inject_replay_drift(X_test_t, 0.05, drift_rate=0.10),
    'Partial (Gage only)': inject_replay_partial(X_test_t, 0.05, feat_idx=[0]),
}

print(f"{'Variant':<28}  {'Stagnation':>11}  {'LSTM-VAE':>9}  {'Trans-VAE':>10}")
print("-" * 65)
results_robust = {}
for name, (X_inj, y_inj, *_) in variants.items():
    stag  = stagnation_score(X_inj, window=6)
    e_l   = lstm_reconstruction_error(lstm_enc,  lstm_dec,  X_inj)
    e_t   = lstm_reconstruction_error(trans_enc, trans_dec, X_inj)
    auc_s = roc_auc_score(y_inj, -stag)
    auc_l = roc_auc_score(y_inj,  e_l)
    auc_t = roc_auc_score(y_inj,  e_t)
    results_robust[name] = (auc_s, auc_l, auc_t)
    print(f"{name:<28}  {auc_s:>11.3f}  {auc_l:>9.3f}  {auc_t:>10.3f}")

# ── Plot: AUC vs noise level ────────────────────────────────────────────
noise_levels = [0.0, 0.05, 0.10, 0.20]
stag_aucs = [results_robust['Exact replay'][0],
             results_robust['Replay+noise σ=0.05'][0],
             results_robust['Replay+noise σ=0.10'][0],
             results_robust['Replay+noise σ=0.20'][0]]
lstm_aucs = [results_robust['Exact replay'][1],
             results_robust['Replay+noise σ=0.05'][1],
             results_robust['Replay+noise σ=0.10'][1],
             results_robust['Replay+noise σ=0.20'][1]]
trans_aucs= [results_robust['Exact replay'][2],
             results_robust['Replay+noise σ=0.05'][2],
             results_robust['Replay+noise σ=0.10'][2],
             results_robust['Replay+noise σ=0.20'][2]]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(noise_levels, stag_aucs,  'b-o',  lw=2, label='Stagnation (proposed)')
ax.plot(noise_levels, lstm_aucs,  'r--s', lw=2, label='LSTM-VAE')
ax.plot(noise_levels, trans_aucs, 'g:^',  lw=2, label='Trans-VAE')
ax.axhline(0.5, color='k', ls='--', alpha=0.3, label='Random')
ax.set_xlabel('Replay Noise σ (fraction of feature std)')
ax.set_ylabel('AUC')
ax.set_title('Stagnation Score Robustness to Adversarial Noise\n(Replay Attack, 5% FDI)')
ax.legend(); ax.grid(alpha=0.3)
ax.set_xlim(-0.01, 0.21); ax.set_ylim(0.4, 1.05)
plt.tight_layout(); plt.savefig('stagnation_robustness.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Partial Replay Robustness Analysis ─────────────────────────────────
# Features: 0=Gage, 1=Crosslevel, 2=Warp62, 3=LProf62, 4=RProf62,
#           5=LAlign62, 6=RAlign62

FEAT_NAMES = ['Gage', 'Crosslevel', 'Warp62', 'LProf62', 'RProf62', 'LAlign62', 'RAlign62']

def inject_replay_partial(X_seq, pct, feat_idx, replay_window=6, seed=SEED):
    """Replay only specified feature indices."""
    rng = np.random.default_rng(seed)
    n_seq, n_t, n_f = X_seq.shape
    n_attack   = max(1, int(n_seq * pct))
    attack_idx = rng.choice(n_seq, n_attack, replace=False)
    X_inj = X_seq.copy()
    y     = np.zeros(n_seq, dtype=int)
    for idx in attack_idx:
        y[idx] = 1
        src_start = int(rng.integers(0, n_t // 2 - replay_window + 1))
        tgt_start = n_t - replay_window
        X_inj[idx, tgt_start:, feat_idx] = X_seq[idx, src_start:src_start + replay_window, feat_idx]
    return X_inj, y

def eval_stagnation(X_inj, y_inj, window=6):
    stag = stagnation_score(X_inj, window=window)
    return roc_auc_score(y_inj, -stag)

def eval_lstm(X_inj, y_inj):
    err = lstm_reconstruction_error(lstm_enc, lstm_dec, X_inj)
    return roc_auc_score(y_inj, err)

def eval_trans(X_inj, y_inj):
    err = lstm_reconstruction_error(trans_enc, trans_dec, X_inj)
    return roc_auc_score(y_inj, err)

# ── Single-feature replay ───────────────────────────────────────────────
print("SINGLE-FEATURE PARTIAL REPLAY (5% FDI)")
print(f"{'Feature':<12}  {'Stagnation':>11}  {'LSTM-VAE':>9}  {'Trans-VAE':>10}")
print("-" * 48)

single_stag, single_lstm, single_trans = [], [], []
for i, name in enumerate(FEAT_NAMES):
    X_inj, y_inj = inject_replay_partial(X_test_t, 0.05, feat_idx=[i])
    a_s = eval_stagnation(X_inj, y_inj)
    a_l = eval_lstm(X_inj, y_inj)
    a_t = eval_trans(X_inj, y_inj)
    single_stag.append(a_s); single_lstm.append(a_l); single_trans.append(a_t)
    print(f"{name:<12}  {a_s:>11.3f}  {a_l:>9.3f}  {a_t:>10.3f}")

# ── Multi-feature combinations ──────────────────────────────────────────
from itertools import combinations

safety_feats = [0, 1, 2]   # Gage, Crosslevel, Warp62 — geometry-critical
combos = list(combinations(safety_feats, 2)) + \
         [(0,1,2)]          # all three safety features

print(f"\nMULTI-FEATURE PARTIAL REPLAY (safety-critical features, 5% FDI)")
print(f"{'Features':<22}  {'Stagnation':>11}  {'LSTM-VAE':>9}  {'Trans-VAE':>10}")
print("-" * 58)

combo_labels, combo_stag, combo_lstm, combo_trans = [], [], [], []
for feat_idx in combos:
    label = '+'.join(FEAT_NAMES[i] for i in feat_idx)
    X_inj, y_inj = inject_replay_partial(X_test_t, 0.05, feat_idx=list(feat_idx))
    a_s = eval_stagnation(X_inj, y_inj)
    a_l = eval_lstm(X_inj, y_inj)
    a_t = eval_trans(X_inj, y_inj)
    combo_labels.append(label); combo_stag.append(a_s)
    combo_lstm.append(a_l);     combo_trans.append(a_t)
    print(f"{label:<22}  {a_s:>11.3f}  {a_l:>9.3f}  {a_t:>10.3f}")

# ── Heatmap: Stagnation AUC by feature scope ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: single-feature bar chart
ax = axes[0]
x  = np.arange(len(FEAT_NAMES))
bars = ax.bar(x, single_stag, color=['steelblue' if s >= 0.7 else 'coral' for s in single_stag],
              alpha=0.85, edgecolor='k', linewidth=0.5)
ax.plot(x, single_lstm,  'r--o', lw=1.5, ms=5, label='LSTM-VAE')
ax.plot(x, single_trans, 'g:^',  lw=1.5, ms=5, label='Trans-VAE')
ax.axhline(0.5, color='k', ls=':', alpha=0.4)
ax.set_xticks(x); ax.set_xticklabels(FEAT_NAMES, rotation=30, ha='right', fontsize=9)
ax.set_ylim(0.4, 1.05); ax.set_ylabel('AUC'); ax.legend(fontsize=8)
ax.set_title('Single-Feature Partial Replay\nStagnation vs Baselines')
ax.grid(axis='y', alpha=0.3)

# Right: multi-feature bar chart
ax = axes[1]
x2 = np.arange(len(combo_labels))
ax.bar(x2, combo_stag, color=['steelblue' if s >= 0.7 else 'coral' for s in combo_stag],
       alpha=0.85, edgecolor='k', linewidth=0.5)
ax.plot(x2, combo_lstm,  'r--o', lw=1.5, ms=5, label='LSTM-VAE')
ax.plot(x2, combo_trans, 'g:^',  lw=1.5, ms=5, label='Trans-VAE')
ax.axhline(0.5, color='k', ls=':', alpha=0.4)
ax.set_xticks(x2); ax.set_xticklabels(combo_labels, rotation=20, ha='right', fontsize=9)
ax.set_ylim(0.4, 1.05); ax.legend(fontsize=8)
ax.set_title('Multi-Feature Partial Replay\n(Safety-Critical Combinations)')
ax.grid(axis='y', alpha=0.3)

plt.suptitle('Stagnation Score Robustness — Partial Replay at 5% FDI', fontweight='bold')
plt.tight_layout()
plt.savefig('partial_replay_robustness.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Summary of Contributions

| Aspect | This Work |
|---|---|
| **Static model** | IF / VAE / VAE+MAD on single-run track geometry |
| **Temporal model** | LSTM-VAE on 28-month repeated inspections |
| **Attack types** | Spike · Multiplicative · Noise · Persistent · One-Shot |
| **Thresholding** | Percentile vs Cross-validation (CV) |
| **Evaluation** | Precision · Recall · F1 · AUC-ROC · McNemar's test |
| **Visualisation** | PCA · t-SNE · Radar · ROC · Temporal signal plots |

> **Key novelty**: The LSTM-VAE learns *how track geometry evolves normally over time*.
> An FDI attack disturbs this temporal pattern — even at low intensity (2%) —
> which the static point-wise models cannot capture.
